<a href="https://colab.research.google.com/github/chitrasavithri176-cmyk/python-ex/blob/main/api_key_ticket.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# ================================================================
# AI-POWERED CUSTOMER SUPPORT TICKET INTELLIGENCE SYSTEM
# Advanced Python Data Structures + Gemini API
# Google Colab - Complete Single Cell Program
# ================================================================

# ================================================================
# 1. INSTALL REQUIRED LIBRARIES
# ================================================================

!pip install -q -U google-genai pydantic python-dotenv


# ================================================================
# 2. IMPORT LIBRARIES
# ================================================================

import os
import json
import csv
import heapq
import time
import logging

from collections import Counter, defaultdict
from typing import Literal

from pydantic import BaseModel, ValidationError
from google import genai
from google.colab import userdata
from google.colab import files


# ================================================================
# 3. LOGGING CONFIGURATION
# ================================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("CustomerSupportAI")


# ================================================================
# 4. GEMINI API CONFIGURATION
# ================================================================

print("=" * 70)
print("AI-POWERED CUSTOMER SUPPORT TICKET INTELLIGENCE SYSTEM")
print("=" * 70)
print()

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = None


# If API key is not stored in Colab Secrets
if not GEMINI_API_KEY:

    from getpass import getpass

    GEMINI_API_KEY = getpass(
        "Enter your Gemini API Key: "
    )


if not GEMINI_API_KEY:

    raise ValueError(
        "Gemini API key was not provided."
    )


# Create Gemini client
client = genai.Client(
    api_key=GEMINI_API_KEY
)


# Current Gemini model
MODEL_NAME = "gemini-3.6-flash"


print("Gemini API configured successfully.")
print("Model:", MODEL_NAME)
print()


# ================================================================
# 5. PYDANTIC STRUCTURED OUTPUT MODEL
# ================================================================

class TicketAnalysis(BaseModel):

    category: Literal[
        "Payment",
        "Technical",
        "Delivery",
        "Product",
        "Account",
        "Other"
    ]

    priority: Literal[
        "Critical",
        "High",
        "Medium",
        "Low"
    ]

    sentiment: Literal[
        "Positive",
        "Neutral",
        "Negative"
    ]

    summary: str

    department: str

    suggested_resolution: str


print("Pydantic structured-output model created.")
print()


# ================================================================
# 6. GEMINI TICKET ANALYSIS FUNCTION
# ================================================================

def analyze_ticket(description, max_retries=3):

    prompt = f"""
You are a customer support ticket classification system.

Analyze the following customer support ticket.

CUSTOMER TICKET:
{description}

Determine the following:

1. Category
2. Priority
3. Sentiment
4. Short summary
5. Responsible department
6. Suggested resolution

Allowed categories:
- Payment
- Technical
- Delivery
- Product
- Account
- Other

Allowed priorities:
- Critical
- High
- Medium
- Low

Allowed sentiments:
- Positive
- Neutral
- Negative

Rules:

- Use only information supported by the ticket.
- Do not invent customer information.
- Give a short and useful summary.
- Give a practical suggested resolution.
- Return structured JSON according to the supplied schema.
"""

    for attempt in range(max_retries):

        try:

            response = client.models.generate_content(

                model=MODEL_NAME,

                contents=prompt,

                config={
                    "response_mime_type": "application/json",
                    "response_schema": TicketAnalysis
                }
            )

            result = TicketAnalysis.model_validate_json(
                response.text
            )

            return result


        except ValidationError as e:

            logger.error(
                "Pydantic validation error: %s",
                e
            )

            if attempt == max_retries - 1:
                raise


        except Exception as e:

            logger.error(
                "Gemini API error: %s",
                e
            )

            if attempt < max_retries - 1:

                wait_time = 2 ** attempt

                print(
                    f"API error. Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:

                raise


# ================================================================
# 7. SAMPLE CUSTOMER SUPPORT TICKETS
# ================================================================

tickets = [

    {
        "ticket_id": "T1001",
        "customer_id": "C101",
        "description":
        "My payment was deducted but my order was not confirmed."
    },

    {
        "ticket_id": "T1002",
        "customer_id": "C102",
        "description":
        "I cannot login to my account after changing my password."
    },

    {
        "ticket_id": "T1003",
        "customer_id": "C103",
        "description":
        "The laptop I received is damaged."
    },

    {
        "ticket_id": "T1004",
        "customer_id": "C101",
        "description":
        "Payment deducted twice for the same order."
    },

    {
        "ticket_id": "T1005",
        "customer_id": "C104",
        "description":
        "My package has not arrived even though the delivery date has passed."
    },

    {
        "ticket_id": "T1006",
        "customer_id": "C105",
        "description":
        "The mobile application crashes whenever I try to open it."
    },

    {
        "ticket_id": "T1007",
        "customer_id": "C106",
        "description":
        "I want to change the email address associated with my account."
    },

    {
        "ticket_id": "T1008",
        "customer_id": "C107",
        "description":
        "I received the wrong product in my order."
    },

    {
        "ticket_id": "T1009",
        "customer_id": "C108",
        "description":
        "The payment was charged but the order confirmation is missing."
    },

    {
        "ticket_id": "T1010",
        "customer_id": "C109",
        "description":
        "I forgot my password and cannot access my account."
    },

    # Intentional duplicate
    {
        "ticket_id": "T1003",
        "customer_id": "C103",
        "description":
        "The laptop I received is damaged."
    }
]


print("=" * 70)
print("INPUT DATA")
print("=" * 70)

print(
    "Total input tickets:",
    len(tickets)
)

print()


# ================================================================
# 8. PYTHON DATA STRUCTURES
# ================================================================

# Dictionary:
# Ticket ID -> Ticket details
ticket_database = {}


# Set:
# Used for duplicate detection
processed_tickets = set()


# Set:
# Unique customers
unique_customers = set()


# List:
# Complete processed records
processed_records = []


# List:
# Duplicate tickets
duplicate_tickets = []


# Nested Dictionary:
# Department -> Category -> Count
department_summary = defaultdict(
    lambda: defaultdict(int)
)


# Ticket history stack
ticket_history = defaultdict(list)


# ================================================================
# 9. PROCESS CUSTOMER SUPPORT TICKETS
# ================================================================

print("=" * 70)
print("PROCESSING TICKETS WITH GEMINI")
print("=" * 70)
print()


for ticket in tickets:

    ticket_id = ticket["ticket_id"]

    customer_id = ticket["customer_id"]

    description = ticket["description"]


    # ------------------------------------------------------------
    # DUPLICATE DETECTION USING SET
    # ------------------------------------------------------------

    if ticket_id in processed_tickets:

        print(
            f"Duplicate ticket detected: {ticket_id}"
        )

        duplicate_tickets.append(
            ticket_id
        )

        continue


    # Add ticket ID to processed set
    processed_tickets.add(
        ticket_id
    )


    # Add customer ID to unique customer set
    unique_customers.add(
        customer_id
    )


    print(
        f"Analyzing ticket {ticket_id}..."
    )


    # ------------------------------------------------------------
    # GEMINI ANALYSIS
    # ------------------------------------------------------------

    try:

        analysis = analyze_ticket(
            description
        )


    except Exception as e:

        print(
            f"Could not process {ticket_id}: {e}"
        )

        continue


    # ------------------------------------------------------------
    # CREATE COMPLETE RECORD
    # ------------------------------------------------------------

    record = {

        "ticket_id":
            ticket_id,

        "customer_id":
            customer_id,

        "description":
            description,

        "category":
            analysis.category,

        "priority":
            analysis.priority,

        "sentiment":
            analysis.sentiment,

        "summary":
            analysis.summary,

        "department":
            analysis.department,

        "suggested_resolution":
            analysis.suggested_resolution
    }


    # ------------------------------------------------------------
    # DICTIONARY STORAGE
    # ------------------------------------------------------------

    ticket_database[
        ticket_id
    ] = record


    # ------------------------------------------------------------
    # LIST STORAGE
    # ------------------------------------------------------------

    processed_records.append(
        record
    )


    # ------------------------------------------------------------
    # NESTED DICTIONARY ANALYTICS
    # ------------------------------------------------------------

    department_summary[
        analysis.department
    ][analysis.category] += 1


    # ------------------------------------------------------------
    # STACK HISTORY
    # ------------------------------------------------------------

    ticket_history[
        ticket_id
    ].append("Created")

    ticket_history[
        ticket_id
    ].append(
        f"Assigned to {analysis.department}"
    )

    ticket_history[
        ticket_id
    ].append(
        f"Priority set to {analysis.priority}"
    )


print()
print("=" * 70)
print("TICKET PROCESSING COMPLETED")
print("=" * 70)

print(
    "Unique tickets:",
    len(ticket_database)
)

print(
    "Unique customers:",
    len(unique_customers)
)

print(
    "Duplicate tickets:",
    len(duplicate_tickets)
)

print()


# ================================================================
# 10. DISPLAY AI ANALYSIS
# ================================================================

print("=" * 70)
print("GEMINI TICKET ANALYSIS")
print("=" * 70)


for ticket_id, ticket in ticket_database.items():

    print()
    print("-" * 70)

    print(
        "Ticket ID:",
        ticket["ticket_id"]
    )

    print(
        "Customer ID:",
        ticket["customer_id"]
    )

    print(
        "Description:",
        ticket["description"]
    )

    print(
        "Category:",
        ticket["category"]
    )

    print(
        "Priority:",
        ticket["priority"]
    )

    print(
        "Sentiment:",
        ticket["sentiment"]
    )

    print(
        "Department:",
        ticket["department"]
    )

    print(
        "Summary:",
        ticket["summary"]
    )

    print(
        "Suggested Resolution:",
        ticket["suggested_resolution"]
    )


# ================================================================
# 11. TUPLE-BASED COMPOSITE KEYS
# ================================================================

ticket_groups = {}


for ticket in processed_records:

    customer_id = ticket[
        "customer_id"
    ]

    category = ticket[
        "category"
    ]

    department = ticket[
        "department"
    ]


    # Tuple composite key
    ticket_key = (
        customer_id,
        category,
        department
    )


    if ticket_key not in ticket_groups:

        ticket_groups[
            ticket_key
        ] = []


    ticket_groups[
        ticket_key
    ].append(
        ticket["ticket_id"]
    )


print()
print("=" * 70)
print("TUPLE COMPOSITE KEY GROUPS")
print("=" * 70)


for key, ticket_ids in ticket_groups.items():

    print(
        key,
        "->",
        ticket_ids
    )


# ================================================================
# 12. HEAP-BASED PRIORITY QUEUE
# ================================================================

priority_rank = {

    "Critical": 1,

    "High": 2,

    "Medium": 3,

    "Low": 4
}


processing_queue = []


# Add tickets to heap
for ticket in processed_records:

    priority = ticket[
        "priority"
    ]

    ticket_id = ticket[
        "ticket_id"
    ]

    rank = priority_rank.get(
        priority,
        4
    )


    heapq.heappush(

        processing_queue,

        (
            rank,
            ticket_id
        )
    )


print()
print("=" * 70)
print("PRIORITY QUEUE PROCESSING")
print("=" * 70)


while processing_queue:

    rank, ticket_id = heapq.heappop(
        processing_queue
    )

    print(
        f"Priority {rank} -> Ticket {ticket_id}"
    )


# ================================================================
# 13. STACK HISTORY
# ================================================================

print()
print("=" * 70)
print("TICKET HISTORY STACK")
print("=" * 70)


for ticket_id, history in ticket_history.items():

    print()
    print(
        f"Ticket {ticket_id}"
    )

    for action in history:

        print(
            "  ->",
            action
        )


# Demonstrate LIFO
if processed_records:

    first_ticket = (
        processed_records[0]["ticket_id"]
    )

    if ticket_history[first_ticket]:

        latest_action = (
            ticket_history[first_ticket][-1]
        )

        print()
        print(
            "Latest action:",
            latest_action
        )


        # Pop = undo
        removed_action = (
            ticket_history[first_ticket].pop()
        )

        print(
            "Undo action:",
            removed_action
        )


# ================================================================
# 14. ANALYTICS
# ================================================================

category_summary = Counter()

priority_summary = Counter()

sentiment_summary = Counter()

department_counts = Counter()

customer_ticket_count = Counter()


for ticket in processed_records:

    category_summary[
        ticket["category"]
    ] += 1

    priority_summary[
        ticket["priority"]
    ] += 1

    sentiment_summary[
        ticket["sentiment"]
    ] += 1

    department_counts[
        ticket["department"]
    ] += 1

    customer_ticket_count[
        ticket["customer_id"]
    ] += 1


# ================================================================
# 15. SET-BASED BUSINESS ANALYTICS
# ================================================================

billing_customers = set()

technical_customers = set()


for ticket in processed_records:

    if ticket["department"].lower() == "billing":

        billing_customers.add(
            ticket["customer_id"]
        )


    if (
        "technical"
        in ticket["department"].lower()
    ):

        technical_customers.add(
            ticket["customer_id"]
        )


# Set intersection
both = (
    billing_customers
    & technical_customers
)


# Set difference
billing_only = (
    billing_customers
    - technical_customers
)


# Set union
either = (
    billing_customers
    | technical_customers
)


# ================================================================
# 16. DISPLAY ANALYTICS
# ================================================================

print()
print("=" * 70)
print("CATEGORY ANALYSIS")
print("=" * 70)


for category, count in category_summary.items():

    print(
        f"{category:<20}: {count}"
    )


print()
print("=" * 70)
print("PRIORITY ANALYSIS")
print("=" * 70)


for priority, count in priority_summary.items():

    print(
        f"{priority:<20}: {count}"
    )


print()
print("=" * 70)
print("SENTIMENT ANALYSIS")
print("=" * 70)


for sentiment, count in sentiment_summary.items():

    print(
        f"{sentiment:<20}: {count}"
    )


print()
print("=" * 70)
print("DEPARTMENT ANALYSIS")
print("=" * 70)


for department, count in department_counts.items():

    print(
        f"{department:<30}: {count}"
    )


print()
print("=" * 70)
print("SET ANALYSIS")
print("=" * 70)

print(
    "Billing customers:",
    billing_customers
)

print(
    "Technical customers:",
    technical_customers
)

print(
    "Customers in both:",
    both
)

print(
    "Billing only:",
    billing_only
)

print(
    "Customers in either:",
    either
)


# ================================================================
# 17. DEPARTMENT + CATEGORY NESTED DICTIONARY
# ================================================================

print()
print("=" * 70)
print("DEPARTMENT-WISE CATEGORY ANALYSIS")
print("=" * 70)


for department, categories in department_summary.items():

    print()
    print(
        department
    )

    print("-" * 40)

    for category, count in categories.items():

        print(
            f"{category:<20}: {count}"
        )


# ================================================================
# 18. MANAGEMENT DATA
# ================================================================

management_data = {

    "total_tickets":
        len(tickets),

    "processed_tickets":
        len(ticket_database),

    "unique_customers":
        len(unique_customers),

    "duplicate_tickets":
        len(duplicate_tickets),

    "categories":
        dict(category_summary),

    "priorities":
        dict(priority_summary),

    "sentiments":
        dict(sentiment_summary),

    "departments":
        dict(department_counts),

    "department_category_analysis":
        {
            department:
                dict(categories)

            for department, categories
            in department_summary.items()
        }
}


# ================================================================
# 19. GEMINI MANAGEMENT SUMMARY
# ================================================================

management_prompt = f"""
You are a customer support analytics manager.

Analyze the following customer support statistics.

DATA:

{json.dumps(management_data, indent=2)}

Create a concise management report.

Include:

1. Significant operational problems
2. Departments requiring attention
3. Potential causes
4. Recommended operational improvements
5. Overall management summary

Rules:

- Do not invent statistics.
- Use only the supplied data.
- Clearly label possible causes as "Possible Cause".
- Keep the report professional.
- Focus on actionable observations.
"""


try:

    management_response = (
        client.models.generate_content(

            model=MODEL_NAME,

            contents=management_prompt
        )
    )

    management_summary = (
        management_response.text
    )


except Exception as e:

    logger.error(
        "Management summary error: %s",
        e
    )

    management_summary = (
        "Management summary could not be generated."
    )


print()
print("=" * 70)
print("GEMINI MANAGEMENT SUMMARY")
print("=" * 70)
print()

print(
    management_summary
)


# ================================================================
# 20. CREATE FINAL JSON REPORT
# ================================================================

final_json_report = {

    "project":
        "AI-Powered Customer Support Ticket Intelligence System",

    "statistics":
        management_data,

    "tickets":
        list(ticket_database.values()),

    "duplicate_tickets":
        duplicate_tickets,

    "ticket_groups":
        {
            str(key):
                value

            for key, value
            in ticket_groups.items()
        },

    "ticket_history":
        dict(ticket_history),

    "management_summary":
        management_summary
}


with open(
    "ticket_report.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_json_report,
        file,
        indent=4,
        ensure_ascii=False
    )


print()
print(
    "Created: ticket_report.json"
)


# ================================================================
# 21. CREATE CSV REPORT
# ================================================================

csv_filename = (
    "processed_tickets.csv"
)


if processed_records:

    fieldnames = [

        "ticket_id",

        "customer_id",

        "description",

        "category",

        "priority",

        "sentiment",

        "summary",

        "department",

        "suggested_resolution"
    ]


    with open(

        csv_filename,

        "w",

        newline="",

        encoding="utf-8"

    ) as file:

        writer = csv.DictWriter(

            file,

            fieldnames=fieldnames
        )


        writer.writeheader()


        writer.writerows(
            processed_records
        )


    print(
        "Created:",
        csv_filename
    )


# ================================================================
# 22. CREATE TEXT MANAGEMENT REPORT
# ================================================================

with open(
    "management_report.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(
        "=" * 70
    )

    file.write

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 20.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.
AI-POWERED CUSTOMER SUPPORT TICKET INTELLIGENCE SYSTEM

Enter your Gemini API Key: ··········
Gemini API configured successfully.
Model: gemini-3.6-flash

Pydantic structured-output model created.

INPUT DATA
Total input tickets: 11

PROCESSING TICKETS WITH GEMINI

Analyzing ticket T1001...


ERROR:CustomerSupportAI:Gemini API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


API error. Retrying in 1 seconds...


ERROR:CustomerSupportAI:Gemini API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


API error. Retrying in 2 seconds...


...

API error. Retrying in 2 seconds...


ERROR:CustomerSupportAI:Gemini API error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}


Could not process T1010: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Duplicate ticket detected: T1003

TICKET PROCESSING COMPLETED
Unique tickets: 8
Unique customers: 9
Duplicate tickets: 1

GEMINI TICKET ANALYSIS

----------------------------------------------------------------------
Ticket ID: T1002
Customer ID: C102
Description: I cannot login to my account after changing my password.
Category: Account
Priority: High
Sentiment: Negative
Department: Account Support
Summary: Customer cannot log in after changing their password.
Suggested Resolution: Send a password reset link and verify the account status.

----------------------------------------------------------------------
Ticket ID: T1003
Customer ID: C103
Description: The laptop I received is damaged.
Category: Delivery
Priority: High
Sentiment: Negative
Department: Returns and Sh

ERROR:CustomerSupportAI:Management summary error: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}



GEMINI MANAGEMENT SUMMARY

Management summary could not be generated.

Created: ticket_report.json
Created: processed_tickets.csv
